# W8C1 Lab: How a Model Chooses the Next Word

Run every cell from the top. **Everything already works.**

Today you will:

1. read the distribution a real model outputs for one prompt
2. run greedy and beam search and compare their scores
3. watch temperature, top-k and top-p reshape that distribution
4. break greedy on purpose, then fix it

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. distilgpt2 is 82M parameters and runs on a laptop CPU in seconds.
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, logging

logging.set_verbosity_error()
torch.manual_seed(0)

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

PROMPT = "The best thing about living in a small town is"
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("prompt:", PROMPT)

## Part 1. The model scores words, it does not pick one

One forward pass gives one number per word in the vocabulary.

In [ ]:
# GIVEN. The raw distribution over the next word.
ids = tokenizer(PROMPT, return_tensors="pt")
with torch.no_grad():
    logits = model(**ids).logits[0, -1]

probs = logits.softmax(dim=-1)
top = probs.topk(10)
words = [tokenizer.decode([i]).strip() or "(space)" for i in top.indices]

print(f"the model scored all {len(probs):,} words in its vocabulary")
print("the ten it likes best:")
for w, p in zip(words, top.values):
    print(f"   {w:<14} {p:.3f}")

plt.figure(figsize=(7, 3))
plt.bar(words, top.values.numpy(), color="#7C2529")
plt.xticks(rotation=45, ha="right"); plt.ylabel("probability")
plt.title("What comes after the prompt?"); plt.tight_layout(); plt.show()

In [ ]:
# ================== TRY IT 1 ==================
# Those ten words hold only part of the probability.
# How much is left over for everything else, and what is in there?
# ==============================================


## Part 2. Greedy and beam search

Two strategies that never sample. Both are deterministic; they disagree about what to optimise.

<img src="images/beam-tree.png" width="760">

In [ ]:
# GIVEN. Greedy takes the argmax at every step. Beam keeps several
# partial sequences alive and scores each one as a whole.
def decode(**kw):
    out = model.generate(**ids, max_new_tokens=40,
                         pad_token_id=tokenizer.eos_token_id,
                         return_dict_in_generate=True, output_scores=True, **kw)
    text = tokenizer.decode(out.sequences[0], skip_special_tokens=True)
    score = out.sequences_scores[0].item() if hasattr(out, "sequences_scores") \
        and out.sequences_scores is not None else None
    return text, score

greedy_text, _ = decode(do_sample=False)
beam_text, beam_score = decode(do_sample=False, num_beams=4)

print("--- greedy ---");            print(greedy_text)
print("\n--- beam, width 4 ---");   print(beam_text)
print(f"\nbeam sequence score (mean log prob per token): {beam_score:.3f}")

In [ ]:
# ================== TRY IT 2 ==================
# Re-run the beam with num_beams=2 and then num_beams=8.
# Does a wider beam keep changing the text, and does its score keep rising?
# ==============================================


## Part 3. The three knobs

Temperature reshapes the whole distribution. Top-k and top-p cut it down before sampling.

<img src="images/temperature.png" width="820">

In [ ]:
# GIVEN. What each knob does to the SAME distribution.
def reshape(logits, temperature=1.0, top_k=0, top_p=1.0):
    z = logits / temperature
    p = z.softmax(dim=-1)
    if top_k:
        cut = p.topk(top_k).values[-1]
        p = torch.where(p >= cut, p, torch.zeros_like(p))
    if top_p < 1.0:
        order = p.argsort(descending=True)
        running = p[order].cumsum(0)
        keep = order[running - p[order] < top_p]
        mask = torch.zeros_like(p, dtype=torch.bool); mask[keep] = True
        p = torch.where(mask, p, torch.zeros_like(p))
    return p / p.sum()

settings = [("plain softmax", {}),
            ("temperature 0.5", {"temperature": 0.5}),
            ("temperature 1.5", {"temperature": 1.5}),
            ("top-k 10", {"top_k": 10}),
            ("top-p 0.9", {"top_p": 0.9})]

for label, kw in settings:
    p = reshape(logits, **kw)
    alive = int((p > 1e-9).sum())
    print(f"{label:<16} top word {p.max():.3f}   words with any chance: {alive:,}")

In [ ]:
# ================== TRY IT 3 ==================
# Print how many words survive top-p at 0.5, 0.9 and 0.99.
# Which of the three would you ship, and what does the smallest one cost you?
# ==============================================


---

## After the break

## Part 4. Greedy is stuck in a loop. Stop it.

A different prompt, the same greedy decoding, and it derails. `repeats` counts words that appear more than once, so you have a number to beat.

In [ ]:
# GIVEN. The loop, and the number you are trying to get down.
LOOP_PROMPT = "In my opinion, the most important part of learning to code is"
loop_ids = tokenizer(LOOP_PROMPT, return_tensors="pt")

def generate(**kw):
    torch.manual_seed(0)
    out = model.generate(**loop_ids, max_new_tokens=60,
                         pad_token_id=tokenizer.eos_token_id, **kw)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def repeats(text):
    w = text.split()
    return len(w) - len(set(w)), len(w)

baseline = generate(do_sample=False)
r, n = repeats(baseline)
print(baseline)
print(f"\nrepeated words: {r} out of {n}")

In [ ]:
# ================== YOUR TURN 1 ==================
# Get the repeat count under 5.
# Every knob from the first hour is available: temperature, top_k,
# top_p, and one you have not met, repetition_penalty, which divides
# the score of any word already generated.
#
# Change ONE thing per run and write the number down.
#
# Expected: the baseline is 41 out of 62. Several settings get under 5,
#           and one of them barely changes the sentence at all.
# =================================================
text = generate(do_sample=True, temperature=1.0, top_k=0)   # <-- edit this line
r, n = repeats(text)
print(text)
print(f"\nrepeated words: {r} out of {n}")

In [ ]:
# ================== YOUR TURN 2 ==================
# Now do it again without sampling.
# Keep do_sample=False, so the output is still perfectly repeatable,
# and still get the repeat count under 5.
#
# Expected: it is possible. Sampling is not the only way to break a loop,
#           and this matters whenever you need the same output twice.
# =================================================
text = generate(do_sample=False)   # <-- add one argument here
r, n = repeats(text)
print(text)
print(f"\nrepeated words: {r} out of {n}")

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   The ten hold 0.841 of the mass between them, and the top word alone holds
#   0.496. That leaves 0.159 spread across the other 50,247 words. A sixth of
#   every draw comes out of a tail you cannot see, which is what top-k and
#   top-p exist to cut.
#
# TRY IT 2
#   The text keeps changing, and the score does NOT keep rising:
#      width 2  -0.541   "It's a place where you can live." repeated
#      width 4  -0.681   the only one that does not loop
#      width 8  -0.475   loops again, on a different phrase
#   A wider beam searches harder for a high-scoring sequence, and a repeated
#   phrase IS high scoring. Beam search does not fix repetition; width 8 makes
#   it worse than width 4. That is the problem Part 4 hands you.
#
# TRY IT 3
#   p = 0.50 keeps    2 words
#   p = 0.90 keeps   20 words
#   p = 0.99 keeps 1,168 words
#   Ship 0.9. At 0.5 the nucleus is two words wide, so on a step where the
#   model is genuinely unsure you have thrown away its uncertainty and the
#   text goes generic. At 0.99 you have kept most of the junk tail back.
#
# YOUR TURN 1
#   baseline, greedy                            41 / 62
#   do_sample=True, temperature=0.7             44 / 65   WORSE than greedy
#   do_sample=True, temperature=1.5              2 / 54   fixed, text degrades
#   do_sample=True, top_k=10                    27 / 64
#   do_sample=True, top_p=0.9                   12 / 59
#   do_sample=False, repetition_penalty=1.2      3 / 64   barely changes the text
#
# YOUR TURN 2
#   text = generate(do_sample=False, repetition_penalty=1.2)   ->   3 / 64
#   text = generate(do_sample=False, repetition_penalty=1.5)   ->   0 / 42
#   The penalty divides the score of every word already generated, so the
#   argmax stops being a fixed point. The decoding stays deterministic: run it
#   twice and you get the same sentence, which sampling can never promise.
#   That is why code and structured output are generated this way.